# Minnie 1822 CAVE query

Nucleus positions, nucleus volume, proofreading status, and cell types from `minnie65_phase3_v1` at materialization 1822.

In [1]:
from pathlib import Path

import standard_transform
from caveclient import CAVEclient

In [2]:
CAVEclient.setup_token("https://global.daf-apis.com") # for minnie

Visit https://global.daf-apis.com/sticky_auth/settings/tokens and copy an existing token or create a new one.
All datastacks are configured to use this server address.
You will not need to specify a server address when initializing a client for configured datastacks in the future.
Setup complete!


In [2]:
DATASTACK = "minnie65_phase3_v1"
VERSION = 1822
OUTPUT_DIR = Path("/workspaces/ConnectsCommonConnectivity/data/EM minnie 1822")

client = CAVEclient(DATASTACK, version=VERSION)
print(client.materialize.get_version_metadata(VERSION)["time_stamp"])

2026-06-27 14:05:22.813147+00:00


## nucleus positions

### cave query

In [3]:
nuc_df = client.materialize.query_table(
    "nucleus_detection_v0",
    split_positions=True,
    desired_resolution=[1, 1, 1],
)[["id", "pt_position_x", "pt_position_y", "pt_position_z", "pt_root_id", "volume"]]

nuc_df = nuc_df[nuc_df["pt_root_id"] != 0]
print(nuc_df.shape)
nuc_df.head()

(129815, 6)


,id,pt_position_x,pt_position_y,pt_position_z,pt_root_id,volume
1,373879,915264,959104,783720,864691136090135607,229.045044
3,201858,587392,854400,1050680,864691135373893678,93.753838
5,408486,980096,977664,1002960,864691135194387242,103.686142
7,598774,1336384,1093888,828520,864691135741608653,31.230034
8,375326,909760,988800,681280,864691135454412650,105.768875


### deduplicate cells

In [4]:
# nucleus_detection_v0 is the raw table: one nucleus can map to several roots and one root to several nuclei.
pairs = nuc_df[["id", "pt_root_id"]].drop_duplicates()
roots_per_nucleus = pairs.groupby("id")["pt_root_id"].nunique()
nuclei_per_root = pairs.groupby("pt_root_id")["id"].nunique()

ambiguous = nuc_df["id"].isin(
    roots_per_nucleus.index[roots_per_nucleus > 1]
) | nuc_df["pt_root_id"].isin(nuclei_per_root.index[nuclei_per_root > 1])

print(f"dropped {int(ambiguous.sum())} / {len(nuc_df)}")
nuc_df = nuc_df[~ambiguous].reset_index(drop=True)

dropped 11351 / 129815


### transform coordinates

In [5]:
tform = standard_transform.minnie_transform_nm()
xyz = (
    tform.apply(nuc_df[["pt_position_x", "pt_position_y", "pt_position_z"]].values)
    / 1000
)

nuc_df[["x_medial-lateral", "y_dorsal-ventral", "z_caudal-rostral"]] = xyz
nuc_df.head()

,id,pt_position_x,pt_position_y,pt_position_z,pt_root_id,volume,x_medial-lateral,y_dorsal-ventral,z_caudal-rostral
0,373879,915264,959104,783720,864691136090135607,229.045044,0.82819,0.638554,0.78372
1,201858,587392,854400,1050680,864691135373893678,93.753838,0.510691,0.505672,1.05068
2,408486,980096,977664,1002960,864691135194387242,103.686142,0.891157,0.662694,1.00296
3,598774,1336384,1093888,828520,864691135741608653,31.230034,1.23596,0.809528,0.82852
4,406597,980160,912640,1083440,864691136908951918,223.584579,0.896888,0.597923,1.08344


### save

In [6]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
nuc_df.drop(columns="volume").to_parquet(
    OUTPUT_DIR / f"minnie_v{VERSION}_nucleus_positions.parquet"
)

## nucleus volume

### save

In [7]:
nuc_df[["id", "pt_root_id", "volume"]].to_parquet(
    OUTPUT_DIR / f"minnie_v{VERSION}_nucleus_volume.parquet"
)

## proofread status

### cave query

In [8]:
proof_df = client.materialize.query_table("proofreading_status_and_strategy")[
    [
        "id",
        "pt_root_id",
        "valid_id",
        "status_axon",
        "status_dendrite",
        "strategy_axon",
        "strategy_dendrite",
    ]
]

# valid_id != pt_root_id means the segment changed after the proofreading assessment.
proof_df["assessment_stale"] = proof_df["valid_id"] != proof_df["pt_root_id"]
print(proof_df.shape)
proof_df.head()

(2334, 8)


,id,pt_root_id,valid_id,status_axon,status_dendrite,strategy_axon,strategy_dendrite,assessment_stale
0,4584,864691135686494647,864691135686494647,True,True,axon_fully_extended,dendrite_extended,False
1,9,864691136812081779,864691136812081779,True,True,axon_interareal,dendrite_extended,False
2,14,864691136195284556,864691136195284556,True,True,axon_fully_extended,dendrite_extended,False
3,18,864691135479404742,864691135479404742,True,True,axon_interareal,dendrite_extended,False
4,19,864691135975539779,864691135975539779,True,True,axon_interareal,dendrite_extended,False


### save

In [13]:
proof_df.to_parquet(OUTPUT_DIR / f"minnie_v{VERSION}_proofreading_status.parquet")

## cell types

### cave query

In [9]:
ct_df = client.materialize.query_table("cell_type_multifeature_combo")[
    ["id", "pt_root_id", "classification_system", "cell_type"]
]
print(ct_df.shape)
ct_df["cell_type"].value_counts()

(69712, 4)


cell_type
L4IT         14625
L6CT         13772
L2IT         11852
L3IT          7613
L6IT          6393
L5IT          4915
PV            3019
L5ET          1747
MC            1481
ITC           1284
NGC            619
DTC            576
L5NP           560
L1             359
NMC            284
AltDTC         206
AltBasket      190
ITCperi        188
ChC             29
Name: count, dtype: Int64

In [14]:
ct_df.head()

,id,pt_root_id,classification_system,cell_type
0,1,864691136010651052,excitatory,L2IT
1,2,864691136287385283,excitatory,L4IT
2,3,864691135446112658,excitatory,L3IT
3,4,864691135695975834,excitatory,L4IT
4,5,864691135104768077,excitatory,L6IT


### save

In [10]:
ct_df.to_parquet(OUTPUT_DIR / f"minnie_v{VERSION}_cell_types.parquet")